# Scenario A · RF 与 GBDT

在 VS Code 的 Remote-SSH 窗口打开 `/mnt/Rank_DML_Scenario_A_Trees`，选择服务器现有的 Python 内核。先运行环境检查，再选择一个 learner 与阶段。

正式运行每个 learner 有 27 个条件，每条件 1,000 次。Stage 1 保存第一阶段预测与诊断，Stage 2 从缓存计算原九种方法。重复运行会从检查点续跑。

In [ ]:
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path
import os
import subprocess
import sys

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents,
                        Path("/mnt/Rank_DML_Scenario_A_Trees"))
             if (p / "experiment_manifest.json").is_file()
             and (p / "spline_core" / "methods.py").is_file()
             and (p / "run.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("请在 VS Code 打开解压后的 Rank_DML_Scenario_A_Trees 文件夹。")
if sys.version_info < (3, 11):
    raise RuntimeError("请切换到 Python 3.11 或 3.12 内核。")
env = os.environ | {"PYTHONUTF8": "1", "PYTHONUNBUFFERED": "1",
                    "PYTHONDONTWRITEBYTECODE": "1"}
for key in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
            "NUMEXPR_NUM_THREADS", "BLIS_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    env[key] = "1"

INSTALL_REQUIREMENTS = False  # 需要安装或补齐版本时改为 True，重新运行本格。
requirements = ROOT / "requirements.txt"

def inventory():
    pending = []
    for line in requirements.read_text(encoding="utf-8-sig").splitlines():
        package, expected = line.split("==")
        try:
            actual = version(package)
        except PackageNotFoundError:
            actual = "缺失"
        print(f"{package:15} {actual:14} 要求 {expected}")
        if actual != expected:
            pending.append(line)
    return pending

pending = inventory()
if pending and INSTALL_REQUIREMENTS:
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(requirements)],
                   cwd=ROOT, env=env, check=True)
    pending = inventory()
ENV_OK = not pending
print(f"当前内核 {sys.executable}")
print("环境检查通过。" if ENV_OK else "依赖有缺失或版本不同。可切换内核，或安装本项目要求的版本。")

## 运行与续跑

`MODE` 选 smoke、pilot 或 main；`LEARNER` 选 rf 或 gbdt；`STAGE` 选 1、2 或 both。确认后把 `START_RUN` 改成 `True`，再运行此格。Main 默认任何样本量或分布均不筛选。中断此格会停止本次任务，已提交的记录会保留。

In [ ]:
import signal
from IPython.display import clear_output

MODE = "smoke"
LEARNER = "rf"
STAGE = "both"
WORKERS = 12
N = None
LAW = None
START_RUN = False

def run_visible(command):
    is_windows = os.name == "nt"
    with subprocess.Popen(
        [sys.executable, "-B", "-u", *command], cwd=ROOT, env=env, text=True,
        encoding="utf-8", errors="replace", stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, start_new_session=not is_windows,
        creationflags=getattr(subprocess, "CREATE_NO_WINDOW", 0) if is_windows else 0,
    ) as process:
        try:
            for line in process.stdout:
                line = line.strip()
                if line:
                    if "%|" in line:
                        clear_output(wait=True)
                    print(line, flush=True)
        except KeyboardInterrupt:
            if process.poll() is None:
                if is_windows:
                    import psutil
                    try:
                        parent = psutil.Process(process.pid)
                        children = parent.children(recursive=True)
                        parent.terminate()
                        for child in children:
                            try:
                                child.terminate()
                            except psutil.NoSuchProcess:
                                pass
                        psutil.wait_procs(children, timeout=5)
                    except psutil.NoSuchProcess:
                        pass
                else:
                    try:
                        os.killpg(process.pid, signal.SIGTERM)
                    except ProcessLookupError:
                        pass
            process.wait()
            print("已停止。本次已提交的数据可用同一组参数续跑。")
            return
        if process.wait():
            raise RuntimeError(f"运行退出码 {process.returncode}。请查看本格最后的输出。")

if not START_RUN:
    print("尚未启动。确认参数后将 START_RUN 改为 True。")
else:
    if not ENV_OK:
        raise RuntimeError("请先完成环境检查。")
    cmd = ["run.py", "--mode", MODE, "--learner", LEARNER,
           "--stage", str(STAGE), "--workers", str(WORKERS), "--resume"]
    if N is not None:
        cmd += ["--n", str(N)]
    if LAW is not None:
        cmd += ["--law", LAW]
    run_visible(cmd)

## 汇总和三张 PDF

完成至少一组 Stage 2 后运行此格。主实验汇总会一并载入 Spline Ridge 正式结果；smoke 和 pilot 使用各自的输出目录。

In [ ]:
REPORT_MODE = "main"
GENERATE_REPORT = False

if GENERATE_REPORT:
    if not ENV_OK:
        raise RuntimeError("请先完成环境检查。")
    run_visible(["report.py", "--mode", REPORT_MODE])
else:
    print("准备好后把 GENERATE_REPORT 改为 True。")